# Overall Project

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =============================================================================
# NCAAB TRAINING DATA GENERATOR
# Scrapes CBS Sports scoreboard for Feb 16–22, 2026
# For each day: pulls games + final scores, builds 27-feature matrix,
# then saves everything to a multi-sheet Excel file.
# =============================================================================


# =============================================================================
# CELL 1 — Install & Import
# =============================================================================

!pip install -q requests beautifulsoup4 pandas numpy vaderSentiment feedparser lxml openpyxl

import requests
import feedparser
import re
import time
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date, timedelta
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')

import shutil

analyzer = SentimentIntensityAnalyzer()

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
}

# Date range: Feb 16 – Feb 22, 2026
START_DATE = date(2026, 2, 16)
END_DATE   = date(2026, 3, 1)
DATE_RANGE = [START_DATE + timedelta(days=i)
              for i in range((END_DATE - START_DATE).days + 1)]

print(f"✅ Setup complete.")
print(f"   Date range: {START_DATE} to {END_DATE} ({len(DATE_RANGE)} days)")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.2 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete.
   Date range: 2026-02-16 to 2026-03-01 (14 days)


## Cell 2

In [3]:
# =============================================================================
# CELL 2 — CBS Sports Scraper
# =============================================================================

def get_games_for_date_cbs(game_date):
    date_str = game_date.strftime('%Y%m%d')
    url      = f"https://www.cbssports.com/college-basketball/scoreboard/FBS/{date_str}/"
    games    = []

    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'lxml')

        game_cards = soup.find_all('div', class_='single-score-card')
        print(f"  {date_str}: {len(game_cards)} game cards found")

        for card in game_cards:
            try:
                # ── 1. away/home abbrevs from data-abbrev ─────────────────────
                abbrev   = card.get('data-abbrev', '')
                at_chunk = abbrev.split('_')[-1]
                if '@' not in at_chunk:
                    continue
                away_abbr, home_abbr = at_chunk.split('@', 1)

                # ── 2. game status ────────────────────────────────────────────
                status_div = card.find('div', class_='game-status')
                status_raw = status_div.get_text(strip=True).lower() \
                             if status_div else ''
                is_final   = 'final' in status_raw
                if not is_final:
                    continue

                # ── 3. get all <tr> rows (skip header row with no class) ──────
                # BS sees class='tiedGame' on both rows pre-JS render
                # Row order from data-abbrev is reliable: row 0 = away, row 1 = home
                all_trs = [tr for tr in card.find_all('tr')
                           if tr.get('class') is not None]

                if len(all_trs) < 2:
                    print(f"    ⚠️  Less than 2 data rows in {abbrev}")
                    continue

                away_row = all_trs[0]
                home_row = all_trs[1]

                # ── 4. extract name and score from a row ──────────────────────
                def get_name(row):
                    a = row.find('a', class_='team-name-link')
                    return a.get_text(strip=True) if a else None

                def get_score(row):
                    td = row.find('td', class_='total')
                    if td:
                        val = td.get_text(strip=True)
                        return int(val) if val.isdigit() else None
                    return None

                away_name  = get_name(away_row)
                away_score = get_score(away_row)
                home_name  = get_name(home_row)
                home_score = get_score(home_row)

                if not away_name or not home_name:
                    continue

                # ── 5. determine winner/loser from scores ─────────────────────
                if away_score is not None and home_score is not None:
                    if home_score > away_score:
                        winner_name, winner_score = home_name, home_score
                        loser_name,  loser_score  = away_name, away_score
                        home_team_won = 1
                    elif away_score > home_score:
                        winner_name, winner_score = away_name, away_score
                        loser_name,  loser_score  = home_name, home_score
                        home_team_won = 0
                    else:
                        # Genuine tie — should not happen in college basketball
                        winner_name = winner_score = None
                        loser_name  = loser_score  = None
                        home_team_won = None
                else:
                    winner_name = winner_score = None
                    loser_name  = loser_score  = None
                    home_team_won = None

                games.append({
                    'date':          game_date.strftime('%Y-%m-%d'),
                    'away_name':     away_name,
                    'away_score':    away_score,
                    'home_name':     home_name,
                    'home_score':    home_score,
                    'status':        'Final',
                    'home_team_won': home_team_won,
                    'winner_name':   winner_name,
                    'winner_score':  winner_score,
                    'loser_name':    loser_name,
                    'loser_score':   loser_score,
                })

            except Exception as e:
                print(f"    ⚠️  {abbrev}: {e}")
                continue

    except Exception as e:
        print(f"  ❌ Failed to fetch {date_str}: {e}")

    print(f"  {date_str}: {len(games)} final games parsed")
    return games

## Cell 3

In [4]:
# =============================================================================
# CELL 3 — Feature Extraction (same 27-feature schema as main program)
# =============================================================================

# ── Keyword dictionaries ──────────────────────────────────────────────────────
INJURY_KEYWORDS    = ['injur','hurt','out for','day-to-day','doubtful','questionable',
                      'sidelined','knee','ankle','concussion','surgery','absence',
                      'missed','will not play','won\'t play']
LINEUP_KEYWORDS    = ['starting lineup','starting five','lineup change',
                      'inserted into the starting','moved to the bench','benched',
                      'starter','rotation change','depth chart','replacing','new starter']
WIN_KEYWORDS       = ['win','victory','beat','defeated','upset','dominant','blowout',
                      'rolled','cruised','overcame','rallied','comeback','unbeaten','streak']
LOSS_KEYWORDS      = ['loss','lost','defeated','fell to','blown out','collapse',
                      'losing streak','skid','slide','struggle','winless']
MOMENTUM_KEYWORDS  = ['hot','on fire','rolling','clicking','momentum','confidence',
                      'energized','surging','impressive run','on a roll','winning streak']
SLUMP_KEYWORDS     = ['slump','cold','struggling','disappointing','slow','inconsistent',
                      'frustrating','skidding','dropped','winless','rough patch']
COACHING_KEYWORDS  = ['coach','head coach','staff','scheme','strategy','adjustment',
                      'system','game plan','coaching staff','fired','hired','contract',
                      'press conference']
RANKING_KEYWORDS   = ['ranked','ranking','top 25','ap poll','net ranking','bracketology',
                      'seed','projection','poll','ballot','moved up','dropped','climbed']
FATIGUE_KEYWORDS   = ['back-to-back','travel','tired','rest','fatigue','load management',
                      'minutes','heavy schedule','third game','quick turnaround']
HOME_AWAY_KEYWORDS = ['home court','home crowd','road game','away game',
                      'hostile environment','sold out','neutral site',
                      'home advantage','visiting']


def keyword_score(text, keywords):
    text_lower = text.lower()
    hits = sum(1 for kw in keywords if kw in text_lower)
    return min(hits / max(len(keywords) * 0.15, 1), 1.0)


def count_keyword_hits(texts, keywords):
    if not texts: return 0.0
    hits = sum(1 for t in texts if any(kw in t.lower() for kw in keywords))
    return hits / len(texts)


def mean_vader(texts):
    if not texts: return 0.0
    return float(np.mean([analyzer.polarity_scores(t)['compound'] for t in texts]))


def fetch_google_news_articles(team_name, max_articles=20, top_links=3):
    query    = requests.utils.quote(f"{team_name} mens college basketball NCAAB")
    url      = f"https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en"
    articles = []
    try:
        feed    = feedparser.parse(url)
        entries = feed.entries[:max_articles]
        for i, entry in enumerate(entries):
            title      = entry.get('title', '')
            source_url = entry.get('link', '')
            full_text  = title
            if i < top_links and source_url:
                try:
                    r = requests.get(source_url, headers=HEADERS,
                                     timeout=10, allow_redirects=True)
                    r.raise_for_status()
                    full_text = f"{title} {r.text}"
                except requests.exceptions.TooManyRedirects:
                    pass
                except requests.exceptions.Timeout:
                    pass
                except Exception:
                    pass
            articles.append({'title': title, 'full_text': full_text,
                              'source_url': source_url, 'source': 'google_news'})
    except Exception:
        pass
    return articles


def fetch_espn_news_articles(team_name, max_articles=15):
    query    = requests.utils.quote(f"{team_name} college basketball")
    url      = (f"https://site.api.espn.com/apis/common/v3/search"
                f"?query={query}&limit={max_articles}&type=article"
                f"&sport=mens-college-basketball")
    articles = []
    try:
        r    = requests.get(url, headers=HEADERS, timeout=12)
        data = r.json()
        for item in data.get('results', [])[:max_articles]:
            title = item.get('headline', item.get('title', ''))
            desc  = item.get('description', item.get('summary', ''))
            articles.append({'title': title, 'full_text': f"{title} {desc}",
                              'source_url': '', 'source': 'espn'})
    except Exception:
        pass
    return articles


def fetch_espn_targeted(team_name, suffix, max_articles=10):
    query    = requests.utils.quote(f"{team_name} {suffix}")
    url      = (f"https://site.api.espn.com/apis/common/v3/search"
                f"?query={query}&limit={max_articles}&type=article"
                f"&sport=mens-college-basketball")
    articles = []
    try:
        r    = requests.get(url, headers=HEADERS, timeout=12)
        data = r.json()
        for item in data.get('results', [])[:max_articles]:
            title = item.get('headline', '')
            desc  = item.get('description', '')
            articles.append({'title': title, 'full_text': f"{title} {desc}",
                              'source_url': '', 'source': 'espn'})
    except Exception:
        pass
    return articles


def extract_team_features(team_name):
    """Extracts 27 named features for a team using news + sentiment analysis."""
    gn_arts     = fetch_google_news_articles(team_name)
    espn_arts   = fetch_espn_news_articles(team_name)
    inj_arts    = fetch_espn_targeted(team_name, 'injury OR injured OR out')
    lineup_arts = fetch_espn_targeted(team_name, 'lineup OR starter OR rotation OR benched')

    all_arts    = gn_arts + espn_arts
    all_texts   = [a['full_text'] for a in all_arts]
    hl_texts    = [a['title']     for a in all_arts]
    gn_texts    = [a['full_text'] for a in gn_arts]
    espn_texts  = [a['full_text'] for a in espn_arts]
    inj_texts   = [a['full_text'] for a in inj_arts]
    ln_texts    = [a['full_text'] for a in lineup_arts]
    total       = len(all_arts)

    all_scores  = ([analyzer.polarity_scores(t)['compound'] for t in all_texts]
                   if all_texts else [0.0])
    recent      = all_texts[-5:] if len(all_texts) >= 5 else all_texts

    star_re     = re.compile(
        r'(star|leading scorer|best player|top player|key player|starting)'
        r'.{0,40}(injur|out|sidelined|miss)', re.IGNORECASE)
    coach_re    = re.compile(
        r'(coach).{0,30}(fired|resign|left|replaced|interim|stepping down)',
        re.IGNORECASE)

    combined_inj = all_texts + inj_texts
    combined_ln  = all_texts + ln_texts

    return {
        'team_name': team_name,
        # [A] Sentiment
        'sent_overall':               round(mean_vader(all_texts), 4),
        'sent_espn':                  round(mean_vader(espn_texts), 4),
        'sent_google':                round(mean_vader(gn_texts), 4),
        'sent_headlines':             round(mean_vader(hl_texts), 4),
        'sent_recent':                round(mean_vader(recent), 4),
        'sent_pct_positive_articles': round(sum(1 for s in all_scores if s >  0.05) / max(total, 1), 4),
        'sent_pct_negative_articles': round(sum(1 for s in all_scores if s < -0.05) / max(total, 1), 4),
        # [B] Injury
        'inj_mention_rate':           round(count_keyword_hits(combined_inj, INJURY_KEYWORDS), 4),
        'inj_severity_score':         round(max((keyword_score(t, INJURY_KEYWORDS) for t in combined_inj), default=0.0), 4),
        'inj_article_count_norm':     round(min(len(inj_arts) / 10.0, 1.0), 4),
        'inj_key_player_flag':        float(any(star_re.search(t) for t in combined_inj)),
        'inj_sentiment':              round(mean_vader(inj_texts), 4),
        # [C] Lineup
        'lineup_change_signal':       round(count_keyword_hits(combined_ln, LINEUP_KEYWORDS), 4),
        'lineup_starter_mention_rate':round(count_keyword_hits(combined_ln, ['starter','starting']), 4),
        'lineup_benched_signal':      round(count_keyword_hits(combined_ln, ['bench','benched','moved to bench']), 4),
        'lineup_roster_instability':  round(min((count_keyword_hits(combined_ln, LINEUP_KEYWORDS) * 0.5 +
                                                 count_keyword_hits(combined_inj, INJURY_KEYWORDS) * 0.5), 1.0), 4),
        'lineup_sentiment':           round(mean_vader(ln_texts), 4),
        # [D] Momentum
        'momentum_win_mention_rate':  round(count_keyword_hits(all_texts, WIN_KEYWORDS), 4),
        'momentum_loss_mention_rate': round(count_keyword_hits(all_texts, LOSS_KEYWORDS), 4),
        'momentum_score':             round(count_keyword_hits(all_texts, MOMENTUM_KEYWORDS), 4),
        'momentum_slump_score':       round(count_keyword_hits(all_texts, SLUMP_KEYWORDS), 4),
        'momentum_net':               round(count_keyword_hits(all_texts, MOMENTUM_KEYWORDS) -
                                            count_keyword_hits(all_texts, SLUMP_KEYWORDS), 4),
        'momentum_win_loss_ratio':    round(count_keyword_hits(all_texts, WIN_KEYWORDS) /
                                            max(count_keyword_hits(all_texts, LOSS_KEYWORDS), 0.01), 4),
        # [E] Context
        'ctx_coaching_mention_rate':  round(count_keyword_hits(all_texts, COACHING_KEYWORDS), 4),
        'ctx_coaching_instability':   float(any(coach_re.search(t) for t in all_texts)),
        'ctx_ranking_mention_rate':   round(count_keyword_hits(all_texts, RANKING_KEYWORDS), 4),
        'ctx_fatigue_signal':         round(count_keyword_hits(all_texts, FATIGUE_KEYWORDS), 4),
        'ctx_home_away_context':      round(count_keyword_hits(all_texts, HOME_AWAY_KEYWORDS), 4),
        # [F] Data quality
        'data_total_articles_norm':   round(min(total / 35.0, 1.0), 4),
        'data_source_diversity':      round(float((len(gn_arts) > 0) + (len(espn_arts) > 0) +
                                                  (len(inj_arts) > 0) + (len(lineup_arts) > 0)) / 4.0, 4),
        'data_confidence':            round((min(total / 35.0, 1.0) * 0.5 +
                                             float((len(gn_arts) > 0) + (len(espn_arts) > 0) +
                                                   (len(inj_arts) > 0) + (len(lineup_arts) > 0)) / 4.0 * 0.5), 4),
    }


## Cell 4

In [5]:
# =============================================================================
# CELL 4 — Main Loop: Iterate Over Each Date
# =============================================================================

all_games_df    = []
processed_teams = {}  # team_name -> feature dict (cache — avoids re-fetching)

print("=" * 65)
print("  NCAAB TRAINING DATA COLLECTION — Feb 16–22, 2026")
print("=" * 65)

for game_date in DATE_RANGE:
    date_str = game_date.strftime('%Y-%m-%d')
    print(f"\n{'─'*65}")
    print(f"  📅 Processing {date_str}")
    print(f"{'─'*65}")

    # ── Step A: Scrape games for this date ────────────────────────────────────
    day_games = get_games_for_date_cbs(game_date)

    if not day_games:
        print(f"  ⚠️  No games found for {date_str}, skipping.")
        continue

    day_games_df = pd.DataFrame(day_games)

    # ── Validate score columns ────────────────────────────────────────────────
    day_games_df['away_score']    = pd.to_numeric(day_games_df['away_score'],    errors='coerce')
    day_games_df['home_score']    = pd.to_numeric(day_games_df['home_score'],    errors='coerce')
    day_games_df['home_team_won'] = pd.to_numeric(day_games_df['home_team_won'], errors='coerce')

    # ── Derive winner/loser columns ───────────────────────────────────────────
    day_games_df['winner_name']  = np.where(day_games_df['home_team_won'] == 1, day_games_df['home_name'], day_games_df['away_name'])
    day_games_df['winner_score'] = np.where(day_games_df['home_team_won'] == 1, day_games_df['home_score'], day_games_df['away_score'])
    day_games_df['loser_name']   = np.where(day_games_df['home_team_won'] == 1, day_games_df['away_name'], day_games_df['home_name'])
    day_games_df['loser_score']  = np.where(day_games_df['home_team_won'] == 1, day_games_df['away_score'], day_games_df['home_score'])

    # ── Cross-check scores ────────────────────────────────────────────────────
    final_games  = day_games_df.dropna(subset=['home_team_won'])
    score_errors = final_games[final_games['winner_score'] <= final_games['loser_score']]
    if not score_errors.empty:
        print(f"  ⚠️  {len(score_errors)} game(s) with score mismatches:")
        for _, row in score_errors.iterrows():
            print(f"      {row['away_name']} {row['away_score']} @ "
                  f"{row['home_name']} {row['home_score']}")

    # ── Display preview ───────────────────────────────────────────────────────
    display_cols = ['date','away_name','away_score','home_name','home_score',
                    'winner_name','winner_score','loser_name','loser_score',
                    'status','home_team_won']
    print(f"\n  Games: {len(day_games_df)} total, {len(final_games)} final")
    #display(day_games_df[display_cols])

    all_games_df.append(day_games_df)

    # ── Step B: Features — use exact names from Step A, use cache ────────────
    # Pull team names directly from the scraped DataFrame rows — no transformation
    teams_today = {}
    for _, row in day_games_df.iterrows():
        home = row['home_name']
        away = row['away_name']
        if pd.notna(home):
            teams_today[home] = home
        if pd.notna(away):
            teams_today[away] = away

    new_teams    = [t for t in teams_today if t not in processed_teams]
    cached_teams = [t for t in teams_today if t in processed_teams]

    print(f"\n  Teams today      : {len(teams_today)}")
    print(f"  ✅ Cached (skip) : {len(cached_teams)}")
    print(f"  🔍 New (fetch)   : {len(new_teams)}")

    if cached_teams:
        print(f"  Reusing cached features for: {', '.join(cached_teams)}")

    for i, team_name in enumerate(new_teams):
        print(f"\n    [{i+1}/{len(new_teams)}] Fetching: {team_name:<40}", end=' ')
        try:
            feats = extract_team_features(team_name)

            # ── Overwrite team_name in feats with the exact name from Step A ──
            # extract_team_features() may internally strip or alter the name;
            # this guarantees the key in processed_teams matches all_games_df exactly
            feats['team_name'] = team_name

            processed_teams[team_name] = feats
            print(f"sent={feats['sent_overall']:+.3f}  "
                  f"momentum={feats['momentum_net']:+.3f}  "
                  f"conf={feats['data_confidence']:.2f}")
        except Exception as e:
            print(f"❌ {e}")
        time.sleep(0.4)

print(f"\n{'='*65}")
print(f"  ✅ Collection complete.")
print(f"     Total games    : {sum(len(d) for d in all_games_df)}")
print(f"     Final games    : {sum(len(d.dropna(subset=['home_team_won'])) for d in all_games_df)}")
print(f"     Unique teams   : {len(processed_teams)}")
print(f"     API calls saved: {sum(len(d) for d in all_games_df) * 2 - len(processed_teams)}")
print(f"{'='*65}")

  NCAAB TRAINING DATA COLLECTION — Feb 16–22, 2026

─────────────────────────────────────────────────────────────────
  📅 Processing 2026-02-16
─────────────────────────────────────────────────────────────────
  20260216: 24 game cards found
  20260216: 24 final games parsed

  Games: 24 total, 24 final

  Teams today      : 48
  ✅ Cached (skip) : 0
  🔍 New (fetch)   : 48

    [1/48] Fetching: East Texas A&M                           sent=+0.205  momentum=+0.000  conf=0.41

    [2/48] Fetching: SE Louisiana                             sent=+0.225  momentum=+0.000  conf=0.41

    [3/48] Fetching: SC State                                 sent=+0.232  momentum=+0.000  conf=0.41

    [4/48] Fetching: Coppin St.                               sent=+0.198  momentum=+0.000  conf=0.41

    [5/48] Fetching: Old Dominion                             sent=+0.165  momentum=+0.000  conf=0.41

    [6/48] Fetching: Louisiana                                sent=+0.219  momentum=+0.000  conf=0.41

    [7

## Cell 4B

In [6]:
# =============================================================================
# CELL 4b — Re-scrape Games Only (Step A only, Step B skipped)
# Run this cell to refresh game scores without re-fetching team features
# =============================================================================

all_games_df = []

print("=" * 65)
print("  NCAAB GAME SCRAPE ONLY — Feb 16–22, 2026")
print("  (Step B / feature extraction skipped)")
print("=" * 65)

for game_date in DATE_RANGE:
    date_str = game_date.strftime('%Y-%m-%d')
    print(f"\n{'─'*65}")
    print(f"  📅 Processing {date_str}")
    print(f"{'─'*65}")

    # ── Step A: Scrape games for this date ────────────────────────────────────
    day_games = get_games_for_date_cbs(game_date)

    if not day_games:
        print(f"  ⚠️  No games found for {date_str}, skipping.")
        continue

    day_games_df = pd.DataFrame(day_games)

    # ── Validate score columns ────────────────────────────────────────────────
    for col in ['winner_score', 'loser_score', 'score_diff']:
        if col in day_games_df.columns:
            day_games_df[col] = pd.to_numeric(day_games_df[col], errors='coerce')

    # ── Cross-check: winner score must always be >= loser score ──────────────
    final_games  = day_games_df.dropna(subset=['winner_name'])
    score_errors = final_games[
        final_games['winner_score'] <= final_games['loser_score']
    ]
    if not score_errors.empty:
        print(f"  ⚠️  {len(score_errors)} game(s) with score mismatches:")
        for _, row in score_errors.iterrows():
            print(f"      {row['winner_name']} {row['winner_score']} vs "
                  f"{row['loser_name']} {row['loser_score']}")

    # ── Display preview ───────────────────────────────────────────────────────
    print(f"\n  Games: {len(day_games_df)} total, {len(final_games)} final")
    display(day_games_df)

    all_games_df.append(day_games_df)

# ── Assemble master games DataFrame ──────────────────────────────────────────
master_games_df = pd.concat(all_games_df, ignore_index=True)

print(f"\n{'='*65}")
print(f"  ✅ Game scrape complete.")
print(f"     Total games  : {len(master_games_df)}")
print(f"     Final games  : {len(master_games_df.dropna(subset=['winner_name']))}")
print(f"     processed_teams cache untouched: {len(processed_teams)} teams")
print(f"{'='*65}")

  NCAAB GAME SCRAPE ONLY — Feb 16–22, 2026
  (Step B / feature extraction skipped)

─────────────────────────────────────────────────────────────────
  📅 Processing 2026-02-16
─────────────────────────────────────────────────────────────────
  20260216: 24 game cards found
  20260216: 24 final games parsed

  Games: 24 total, 24 final


,date,away_name,away_score,home_name,home_score,status,home_team_won,winner_name,winner_score,loser_name,loser_score
0,2026-02-16,SE Louisiana,53,East Texas A&M,70,Final,1,East Texas A&M,70,SE Louisiana,53
1,2026-02-16,Coppin St.,59,SC State,57,Final,0,Coppin St.,59,SC State,57
2,2026-02-16,Louisiana,72,Old Dominion,83,Final,1,Old Dominion,83,Louisiana,72
3,2026-02-16,Colgate,58,Boston U.,85,Final,1,Boston U.,85,Colgate,58
4,2026-02-16,Bethune-Cook.,86,Jackson St.,91,Final,1,Jackson St.,91,Bethune-Cook.,86
5,2026-02-16,Syracuse,64,Duke,101,Final,1,Duke,101,Syracuse,64
6,2026-02-16,Grambling St.,63,Prairie View,68,Final,1,Prairie View,68,Grambling St.,63
7,2026-02-16,Howard,91,Delaware St.,59,Final,0,Howard,91,Delaware St.,59
8,2026-02-16,Morgan St.,76,NC Central,80,Final,1,NC Central,80,Morgan St.,76
9,2026-02-16,Miss Valley St.,55,Alabama St.,92,Final,1,Alabama St.,92,Miss Valley St.,55



─────────────────────────────────────────────────────────────────
  📅 Processing 2026-02-17
─────────────────────────────────────────────────────────────────
  20260217: 30 game cards found
  20260217: 30 final games parsed

  Games: 30 total, 30 final


,date,away_name,away_score,home_name,home_score,status,home_team_won,winner_name,winner_score,loser_name,loser_score
0,2026-02-17,Boston College,72,Florida St.,80,Final,1,Florida St.,80,Boston College,72
1,2026-02-17,Gardner-Webb,66,Charleston So.,75,Final,1,Charleston So.,75,Gardner-Webb,66
2,2026-02-17,C. Michigan,54,E. Michigan,66,Final,1,E. Michigan,66,C. Michigan,54
3,2026-02-17,Michigan,91,Purdue,80,Final,0,Michigan,91,Purdue,80
4,2026-02-17,N. Illinois,72,Buffalo,70,Final,0,N. Illinois,72,Buffalo,70
5,2026-02-17,Villanova,92,Xavier,89,Final,0,Villanova,92,Xavier,89
6,2026-02-17,Akron,90,W. Michigan,73,Final,0,Akron,90,W. Michigan,73
7,2026-02-17,Ball St.,57,Ohio,69,Final,1,Ohio,69,Ball St.,57
8,2026-02-17,Kent St.,78,Bowling Green,71,Final,0,Kent St.,78,Bowling Green,71
9,2026-02-17,Louisville,85,SMU,95,Final,1,SMU,95,Louisville,85



─────────────────────────────────────────────────────────────────
  📅 Processing 2026-02-18
─────────────────────────────────────────────────────────────────
  20260218: 59 game cards found
  20260218: 59 final games parsed

  Games: 59 total, 59 final


...


─────────────────────────────────────────────────────────────────
  📅 Processing 2026-02-28
─────────────────────────────────────────────────────────────────
  20260228: 143 game cards found
  20260228: 143 final games parsed

  Games: 143 total, 143 final


,date,away_name,away_score,home_name,home_score,status,home_team_won,winner_name,winner_score,loser_name,loser_score
0,2026-02-28,Boston U.,68,American,65,Final,0,Boston U.,68,American,65
1,2026-02-28,Colorado,62,Houston,102,Final,1,Houston,102,Colorado,62
2,2026-02-28,Florida St.,80,Ga. Tech,71,Final,0,Florida St.,80,Ga. Tech,71
3,2026-02-28,Iowa,69,Penn St.,71,Final,1,Penn St.,71,Iowa,69
4,2026-02-28,Loyola-Md.,76,Holy Cross,62,Final,0,Loyola-Md.,76,Holy Cross,62
...,...,...,...,...,...,...,...,...,...,...,...
138,2026-02-28,Cal Poly,64,UC San Diego,80,Final,1,UC San Diego,80,Cal Poly,64
139,2026-02-28,Grand Canyon,69,Utah St.,74,Final,1,Utah St.,74,Grand Canyon,69
140,2026-02-28,Nevada,83,UNLV,85,Final,1,UNLV,85,Nevada,83
141,2026-02-28,Gonzaga,59,Saint Mary's,70,Final,1,Saint Mary's,70,Gonzaga,59



─────────────────────────────────────────────────────────────────
  📅 Processing 2026-03-01
─────────────────────────────────────────────────────────────────
  20260301: 23 game cards found
  20260301: 23 final games parsed

  Games: 23 total, 23 final


,date,away_name,away_score,home_name,home_score,status,home_team_won,winner_name,winner_score,loser_name,loser_score
0,2026-03-01,La Salle,64,Davidson,71,Final,1,Davidson,71,La Salle,64
1,2026-03-01,North Texas,62,UAB,58,Final,0,North Texas,62,UAB,58
2,2026-03-01,Rutgers,69,Maryland,65,Final,0,Rutgers,69,Maryland,65
3,2026-03-01,Tulane,62,South Florida,90,Final,1,South Florida,90,Tulane,62
4,2026-03-01,UIC,63,Indiana St.,79,Final,1,Indiana St.,79,UIC,63
5,2026-03-01,Quinnipiac,67,Canisius,63,Final,0,Quinnipiac,67,Canisius,63
6,2026-03-01,Purdue,74,Ohio St.,82,Final,1,Ohio St.,82,Purdue,74
7,2026-03-01,Charlotte,76,FAU,77,Final,1,FAU,77,Charlotte,76
8,2026-03-01,Iona,69,Manhattan,65,Final,0,Iona,69,Manhattan,65
9,2026-03-01,Memphis,68,East Carolina,84,Final,1,East Carolina,84,Memphis,68



  ✅ Game scrape complete.
     Total games  : 691
     Final games  : 691
     processed_teams cache untouched: 365 teams


## Cell 5

In [7]:
# =============================================================================
# CELL 5 — Assemble Final DataFrames
# =============================================================================

# ── Master games table ────────────────────────────────────────────────────────
master_games_df = pd.concat(all_games_df, ignore_index=True)

# ── Master features table ─────────────────────────────────────────────────────
master_features_df = pd.DataFrame(list(processed_teams.values()))

META_COLS    = ['team_name', 'date']
numeric_cols = [c for c in master_features_df.columns if c not in META_COLS]
for col in numeric_cols:
    master_features_df[col] = pd.to_numeric(master_features_df[col],
                                             errors='coerce').fillna(0.0)

# ── Training table: one row per game with both teams' features joined ─────────
# Join home team features
home_feats = master_features_df.add_prefix('home_').rename(
    columns={'home_team_name': 'home_name'})
away_feats = master_features_df.add_prefix('away_').rename(
    columns={'away_team_name': 'away_name'})

training_df = master_games_df.merge(home_feats, on='home_name', how='left')
training_df = training_df.merge(away_feats,     on='away_name', how='left')

# Drop rows with no outcome (non-final games)
training_df = training_df.dropna(subset=['home_team_won']).reset_index(drop=True)
training_df['home_team_won'] = training_df['home_team_won'].astype(int)

print(f"✅ DataFrames assembled.")
print(f"   master_games_df   : {master_games_df.shape}")
print(f"   master_features_df: {master_features_df.shape}")
print(f"   training_df        : {training_df.shape}  ← use this for ML training")

✅ DataFrames assembled.
   master_games_df   : (691, 11)
   master_features_df: (365, 32)
   training_df        : (691, 73)  ← use this for ML training


## Cell 6

In [8]:
# =============================================================================
# CELL 6 — Save to CSV
# =============================================================================

import os

OUTPUT_PATH = '/content/drive/MyDrive/Colab Notebooks'

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Assemble master games DataFrame ──────────────────────────────────────────
master_games_df = pd.concat(all_games_df, ignore_index=True)

# ── Assemble master features DataFrame from cache ────────────────────────────
master_features_df = pd.DataFrame(list(processed_teams.values()))
META_COLS    = ['team_name']
numeric_cols = [c for c in master_features_df.columns if c not in META_COLS]
for col in numeric_cols:
    master_features_df[col] = pd.to_numeric(master_features_df[col], errors='coerce').fillna(0.0)

# ── Save ──────────────────────────────────────────────────────────────────────
games_path    = os.path.join(OUTPUT_PATH, 'ncaab_games_2.csv')
features_path = os.path.join(OUTPUT_PATH, 'ncaab_team_features_2.csv')

master_games_df.to_csv(games_path,    index=False)
master_features_df.to_csv(features_path, index=False)

print(f"✅ Saved:")
print(f"   Games CSV    → {games_path}    ({master_games_df.shape[0]} rows × {master_games_df.shape[1]} cols)")
print(f"   Features CSV → {features_path} ({master_features_df.shape[0]} rows × {master_features_df.shape[1]} cols)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Saved:
   Games CSV    → /content/drive/MyDrive/Colab Notebooks/ncaab_games_2.csv    (691 rows × 11 cols)
   Features CSV → /content/drive/MyDrive/Colab Notebooks/ncaab_team_features_2.csv (365 rows × 32 cols)


In [9]:
master_games_df.shape

(691, 11)

In [10]:
# =============================================================================
# CELL 5 — Assemble Final DataFrames
# =============================================================================

master_games_df = pd.concat(all_games_df, ignore_index=True)

# ── Force numeric on score columns ───────────────────────────────────────────
for col in ['away_score', 'home_score', 'winner_score', 'loser_score']:
    if col in master_games_df.columns:
        master_games_df[col] = pd.to_numeric(master_games_df[col], errors='coerce')

# ── score_diff = home_score minus away_score ──────────────────────────────────
master_games_df['score_diff'] = master_games_df['home_score'] - master_games_df['away_score']

# ── Summary ───────────────────────────────────────────────────────────────────
final_games = master_games_df.dropna(subset=['winner_name'])

print(f"✅ master_games_df assembled.")
print(f"   Total rows        : {len(master_games_df)}")
print(f"   Final games       : {len(final_games)}")
print(f"   Score diff range  : {final_games['score_diff'].min():.0f} "
      f"to {final_games['score_diff'].max():.0f} pts")
print(f"   Home win rate     : {(final_games['home_team_won'] == 1).mean():.1%}")
print(f"\n   Sample:")
display(master_games_df.head(10))


# =============================================================================
# CELL 6 — Save Games CSV to Google Drive
# =============================================================================

import os
from google.colab import drive

drive.mount('/content/drive')

OUTPUT_PATH = '/content/drive/MyDrive/Colab Notebooks'
games_path  = os.path.join(OUTPUT_PATH, 'ncaab_games_2.csv')

master_games_df.to_csv(games_path, index=False)

print(f"✅ Saved:")
print(f"   Games CSV → {games_path}")
print(f"              {len(master_games_df)} rows × {master_games_df.shape[1]} cols")
print(f"\n   Columns saved:")
for col in master_games_df.columns:
    non_null = master_games_df[col].notna().sum()
    print(f"     • {col:<25} ({non_null} non-null / {len(master_games_df)} rows)")

✅ master_games_df assembled.
   Total rows        : 691
   Final games       : 691
   Score diff range  : -44 to 41 pts
   Home win rate     : 61.2%

   Sample:


,date,away_name,away_score,home_name,home_score,status,home_team_won,winner_name,winner_score,loser_name,loser_score,score_diff
0,2026-02-16,SE Louisiana,53,East Texas A&M,70,Final,1,East Texas A&M,70,SE Louisiana,53,17
1,2026-02-16,Coppin St.,59,SC State,57,Final,0,Coppin St.,59,SC State,57,-2
2,2026-02-16,Louisiana,72,Old Dominion,83,Final,1,Old Dominion,83,Louisiana,72,11
3,2026-02-16,Colgate,58,Boston U.,85,Final,1,Boston U.,85,Colgate,58,27
4,2026-02-16,Bethune-Cook.,86,Jackson St.,91,Final,1,Jackson St.,91,Bethune-Cook.,86,5
5,2026-02-16,Syracuse,64,Duke,101,Final,1,Duke,101,Syracuse,64,37
6,2026-02-16,Grambling St.,63,Prairie View,68,Final,1,Prairie View,68,Grambling St.,63,5
7,2026-02-16,Howard,91,Delaware St.,59,Final,0,Howard,91,Delaware St.,59,-32
8,2026-02-16,Morgan St.,76,NC Central,80,Final,1,NC Central,80,Morgan St.,76,4
9,2026-02-16,Miss Valley St.,55,Alabama St.,92,Final,1,Alabama St.,92,Miss Valley St.,55,37


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Saved:
   Games CSV → /content/drive/MyDrive/Colab Notebooks/ncaab_games_2.csv
              691 rows × 12 cols

   Columns saved:
     • date                      (691 non-null / 691 rows)
     • away_name                 (691 non-null / 691 rows)
     • away_score                (691 non-null / 691 rows)
     • home_name                 (691 non-null / 691 rows)
     • home_score                (691 non-null / 691 rows)
     • status                    (691 non-null / 691 rows)
     • home_team_won             (691 non-null / 691 rows)
     • winner_name               (691 non-null / 691 rows)
     • winner_score              (691 non-null / 691 rows)
     • loser_name                (691 non-null / 691 rows)
     • loser_score               (691 non-null / 691 rows)
     • score_diff                (691 non-null / 691 rows)
